In [1]:
pip install langchain langchain-openai langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 129.7 kB/s eta 0:00:000:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.0 MB/s eta 0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.0 MB/s eta 0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 9.3 MB/s eta 0:00:00 eta 0:00:01
 

In [6]:
import os 
from getpass import getpass 
from langsmith import traceable 
from langchain_openai import AzureChatOpenAI 
from langchain_core.prompts import PromptTemplate 
from langchain_core.messages import HumanMessage

# ---------- 1) Force prompts for secrets (one-time) ---------- 
os.environ.pop("AZURE_OPENAI_API_KEY", None) 
os.environ.pop("LANGCHAIN_API_KEY", None) 
os.environ["AZURE_OPENAI_API_KEY"] = getpass("Enter your AZURE_OPENAI_API_KEY: ") 
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter your LANGCHAIN_API_KEY: ") 
os.environ.setdefault("LANGCHAIN_PROJECT", "Azure-LLM-Demo") 

# Optional Azure environment settings 
os.environ.setdefault("OPENAI_API_TYPE", "azure") 
os.environ.setdefault("OPENAI_API_VERSION", "2025-01-01-preview") 
# Sanity check (no secrets printed) 
print( "Tracing on?:", os.getenv("LANGCHAIN_TRACING_V2") == "true", 
      "| Project:", os.getenv("LANGCHAIN_PROJECT"), 
      "| Has LangSmith key?:", bool(os.getenv("LANGCHAIN_API_KEY")) ) 

# ---------- 2) Azure Chat Model ---------- # 
# The endpoint URL is written as three separate strings so that no part of it 
# # can be broken by line wrapping. Python joins them into a single URL. 
AZURE_ENDPOINT = ( "https://openai-api-management-gw.azure-api.net" 
                  "/openaiprodtest-clone/deployments/gpt-5-mini/chat/completions" 
                  "?api-version=2024-12-01-preview" ) 

llm = AzureChatOpenAI( azure_endpoint=AZURE_ENDPOINT, api_version="2024-12-01-preview", deployment_name="gpt-5-mini", ) 

# ---------- 3) Prompt Template ---------- 
prompt = PromptTemplate.from_template( "Answer clearly and concisely:\n{question}" ) 

# ---------- 4) Traceable function ---------- 
@traceable(name="SimpleAgentTrace") 
def simple_agent_response(question: str) -> str: 
    formatted_prompt = prompt.format(question=question) 
    response = llm.invoke( [HumanMessage(content=formatted_prompt)] 
    ) 
    
    return response.content 

# ---------- 5) Run ---------- 
if __name__ == "__main__": 
    q = "What are the key benefits of observability in AI agents?" 
    print("Question:", q)
    answer = simple_agent_response(q) 
    print("\nAnswer:\n", answer)

Tracing on?: True | Project: Azure-LLM-Demo | Has LangSmith key?: True
Question: What are the key benefits of observability in AI agents?

Answer:
 - Faster debugging and root-cause analysis: detailed traces and metrics let developers quickly identify where an agent failed.
- Continuous performance monitoring: track latency, accuracy, and throughput to keep agents meeting SLAs.
- Drift and anomaly detection: surface data/model drift, distribution shifts, and unexpected behaviors early.
- Safety and robustness: detect unsafe or out-of-distribution actions and enable timely mitigation.
- Explainability and transparency: logs and traces support understanding of agent decisions for users and stakeholders.
- Auditability and compliance: recorded evidence of inputs, outputs, and model versions for regulatory and governance needs.
- Faster iteration and deployment: empirical feedback shortens development cycles and improves model tuning.
- Cost and resource optimization: visibility into resou